In [177]:
import json
from pathlib import Path
import typer
import random
from tqdm import tqdm
import re


In [178]:
# Load functions

def load_rules(rules_filename):
    with open(rules_filename, "r", encoding="utf-8") as f:
        rules_data = json.load(f)

    all_rules = []
    for section in rules_data:
        if section is not None:  # Check if section is not None
            for paragraph in section:
                if paragraph is not None:  # Check if paragraph is not None
                    all_rules.extend(paragraph)

    rules_with_fst = [
        rule
        for rule in all_rules
        if "fst_rule" in rule and rule["fst_rule"] is not None
    ]

    rules_with_unimorph = [
        rule
        for rule in all_rules
        if "unimorph" in rule and rule["unimorph"] is not None
    ]
    return all_rules, rules_with_fst, rules_with_unimorph

def load_dictionary(dict_path,reverse=False):
    try:
        with open(dict_path, "r", encoding="utf-8") as f:
            dictionary = json.load(f)
            if reverse:
                # reverse the dictionary to map from language to English
                dictionary = {v.lower(): k.lower() for k, v in dictionary.items()}
            return dictionary
    except FileNotFoundError:
        print(
            f"Dictionary file not found at {dict_path}. Please provide a valid dictionary file."
        )
        return {}

def load_dictionary_with_meta(dict_path):
    try:
        with open(dict_path, "r", encoding="utf-8") as f:
            dictionary = json.load(f)
            return dictionary
    except FileNotFoundError:
        print(
            f"Dictionary file not found at {dict_path}. Please provide a valid dictionary file."
        )
        return {}


In [179]:
# Filter functions

def filter_dict_by_pos(dictionary_with_metadata, target_pos):
    """
    Filters a dictionary to include only words that are predominantly of the specified parts of speech (POS
    categories).
    Args:
        dictionary_with_metadata (dict): A dictionary where keys are words and values are metadata including senses
        target_pos (list[str]): List of POS categories to filter by (e.g. ["NOUN", "VERB"]).
    Returns:
        dict: Filtered dictionary containing only words that are predominantly of the specified POS categories.
    """
    filtered_dict = {}

    for word, word_metadata in dictionary_with_metadata.items():
        senses = word_metadata.get("senses", [])
        if not senses:
            continue

        pos_like = False
        count = 0
        for sense in senses:
            spacy_analysis = sense.get("spacy_analysis")
            if not spacy_analysis:
                continue
                

            pos = spacy_analysis.get("head_pos")
            if pos in target_pos:
                count += 1

        if count/len(senses) >= 0.5:  # at least half the senses are POS-like
            pos_like = True

        if pos_like:
            filtered_dict[word] = dictionary_with_metadata.get(word, word_metadata)

    return filtered_dict

def find_lrl_words_by_pos(sentence_text, pos_dicts, target_pos):
    """
    Finds which *target-language words* in a sentence appear in your filtered POS dictionaries.

    Args:
        sentence_text (str): Sentence in the target language (LRL).
        pos_dicts (dict): Mapping POS → dictionary of LRL words (keys).
        target_pos (list[str]): POS categories to match (e.g. ["NOUN", "VERB"]).

    Returns:
        dict: {pos_tag: [matched_words]} of words found in the sentence.
    """
    # Normalize the sentence (lowercase, basic punctuation removal)
    sentence_clean = re.sub(r"[^\w\s'=*-]", " ", sentence_text.lower())
    matches_by_pos = {}

    for pos_tag in target_pos:
        pos_words = set(pos_dicts.get(pos_tag, {}).keys())
        matched = set()

        # Prioritize longer multiword entries first
        for word in sorted(pos_words, key=len, reverse=True):
            # Use regex to match exact word boundaries (handles = and * prefixes too)
            pattern = rf"(?<!\S){re.escape(word.lower())}(?!\S)"
            if re.search(pattern, sentence_clean):
                matched.add(word)

        if matched:
            matches_by_pos[pos_tag] = sorted(matched, key=len, reverse=True)

    return matches_by_pos

def get_random_words_from_dict(pos_dicts, pos_to_replace, no_of_random_nouns):
    # Get a random pos word from the dictionary of selected pos
    random_words_of_selected_pos = {}
    for pos in pos_to_replace:
        pos_dict = pos_dicts.get(pos, {})
        random_words = random.sample(
            list(pos_dict.keys()),
            min(no_of_random_nouns, len(pos_dict))
        )
        for word in random_words:
            random_words_of_selected_pos[word] = pos_dict[word]

    return random_words_of_selected_pos



In [180]:
# 

def get_lrl_words_meta(lrl_words_by_pos, spacy_info, dictionary, sentence):
        gender = None
        plural = None
        case = None
        morph = None
        if lrl_words_by_pos == {}:
            print("No words found in the sentence to replace.")


        else:
            print(f"Words to be replaced: {lrl_words_by_pos}")
            translation_words_to_be_replaced = {}
            for pos_tag, words in lrl_words_by_pos.items():
                for word in words:
                    word = word.strip().lower()
                    word_translation = dictionary.get(word, "No translation available")
                    translation_words_to_be_replaced[word] = word_translation

            if translation_words_to_be_replaced == {}:
                print(f"Warning: No translation found for words {list(lrl_words_by_pos)}")
            else:
                # Clean translation, split by comma and / , remove bracket
                translation_words_to_be_replaced = {
                    word: translation.split(",")[0]
                    .split("/")[0]
                    .split("(")[0]
                    .strip()
                    for word, translation in translation_words_to_be_replaced.items()
                }
            print(f"Translation: {translation_words_to_be_replaced}")

            # Check if each selected word's translation is present in the target sentence
            # TODO: Fix naive search, this can result in words infixed in others.
            words_present = []
            for word,word_translation in translation_words_to_be_replaced.items():
                if word_translation in sentence["translated_sentence"]:
                    print(f"{word_translation} ({word}) present in sentence")
                    words_present.append(word)
            # Randomly select one word to be replaced from present, or else from all
            if words_present!=[]:
                word_to_be_replaced = random.choice(words_present)
            else:
                word_to_be_replaced = random.choice(list(translation_words_to_be_replaced.keys()))
            translation_words_to_be_replaced = translation_words_to_be_replaced[word_to_be_replaced]
            print(f"Selected word to be replaced: {word_to_be_replaced} - {translation_words_to_be_replaced}")
            # Get spacy POS mapping for the sentence
            word_stem_info = None
            for word_info in spacy_info:
                if word_info["lemma"] == translation_words_to_be_replaced:
                    word_stem_info = word_info
                    break

            if word_stem_info:
                print(f"Found word stem info: {word_stem_info}")
                morph = word_stem_info.get("morph", None)
                if morph:
                    plural = morph.get("Number", None)
                    if plural == "Plur":
                        print(f"The word '{word_to_be_replaced}' is plural.")
                    else:
                        print(f"The word '{word_to_be_replaced}' is singular.")
                    case = morph.get("Case", None)
                    if case:
                        print(
                            f"The word '{word_to_be_replaced}' is in the case: {case}."
                        )
                    gender = morph.get("Gender", None)
                    if gender:
                        print(
                            f"The word '{word_to_be_replaced}' is of gender {gender}."
                        )
            else:
                print(f"No word stem info found for: {translation_words_to_be_replaced}")

            # print(f"Morphology Info: {morphology_info}")
        return gender, plural, case


# Rule select function

In [181]:
import torch
rule_embeddings = torch.load("rule_embeddings.pt")

In [182]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# load once globally
model = SentenceTransformer("all-MiniLM-L6-v2")

def select_rules(
    all_rules,
    pos=None,
    gender=None,
    number=None,
    case=None,
    tense=None,
    top_k=10,
    w_embed=0.6,
    w_overlap=0.4,
):
    """
    Hybrid rule selection: combines embedding similarity and tag overlap.

    Args:
        all_rules (list[dict]): full list of rule dicts.
        pos (str): UPOS of the word to replace (e.g. "NOUN", "VERB").
        gender, number, case, tense: current morphological features.
        top_k (int): number of rules to return.
        w_embed, w_overlap (float): weights for similarity fusion.

    Returns:
        str: formatted list of selected rule descriptions.
    """

    # 1️⃣ Build a rich feature string
    features = " ".join(
        filter(None, [pos, gender, number, case, tense])
    ).strip()
    if not features:
        return "No specific rules apply."

    # 2️⃣ Build candidate rule texts
    rule_texts, rule_objs = [], []
    for rule in all_rules:
        text_parts = [
            str(rule.get("description") or ""),
            str(rule.get("condition") or ""),
            " ".join(rule.get("grammar_tags") or []),
        ]
        rule_texts.append(" ".join(text_parts))
        rule_objs.append(rule)

    # 3️⃣ Encode embeddings
    feat_emb = model.encode(features, convert_to_tensor=True)
    rule_embs = rule_embeddings  # precomputed embeddings
    
    # 4️⃣ Compute embedding similarity
    sims = util.cos_sim(feat_emb, rule_embs)[0].cpu().numpy()

    # 5️⃣ Compute tag overlap score (symbolic match)
    feat_tags = set(features.lower().split())
    overlap_scores = []
    for rule in rule_objs:
        rule_tags = set(t.lower() for t in (rule.get("grammar_tags") or []))
        overlap = len(feat_tags & rule_tags)
        overlap_scores.append(overlap / max(1, len(rule_tags)))

    overlap_scores = np.array(overlap_scores)

    # 6️⃣ Combine scores
    final_scores = w_embed * sims + w_overlap * overlap_scores

    # 7️⃣ Rank & select
    top_idx = np.argsort(final_scores)[::-1][:top_k]
    top_rules = [rule_objs[i] for i in top_idx]

    # 8️⃣ Format output
    return "\n".join(f"- {r['description']}" for r in top_rules) or "No specific rules apply."


In [183]:
# Rule selection function naive
def select_rules_naive(all_rules, gender=None, number=None, case=None, tense=None):
    def has(desc, kw): return kw in desc.lower()

    selected = []
    def add(pred): selected.extend([r for r in all_rules if pred(r)])

    if gender == "Masc": add(lambda r: has(r["description"], "masculine"))
    if gender == "Fem":  add(lambda r: has(r["description"], "feminine"))
    if number == "Plur": add(lambda r: has(r["description"], "plural"))
    if case == "Nom":    add(lambda r: has(r["description"], "nominative"))
    if case == "Acc":    add(lambda r: has(r["description"], "accusative"))
    if case == "Dat":    add(lambda r: has(r["description"], "dative"))
    if case == "Gen":    add(lambda r: has(r["description"], "genitive"))
    if case == "Loc":    add(lambda r: has(r["description"], "locative"))
    if case == "Abl":    add(lambda r: has(r["description"], "ablative"))
    if case == "Voc":    add(lambda r: has(r["description"], "vocative"))
    if tense == "Past":  add(lambda r: has(r["description"], "past"))
    if tense == "Pres":  add(lambda r: has(r["description"], "present"))
    if tense == "Fut":   add(lambda r: has(r["description"], "future"))

    # dedupe & cap
    unique = {}
    for r in selected:
        unique.setdefault(r["description"], r)
    capped = list(unique.values())[:50]  # keep context small
    return "\n".join(f"- {r['description']}" for r in capped) or "No specific rules apply."


# Driver code

In [184]:
filename = "kalamang.pdf"
pos_to_replace = "NOUN,PNOUN"  # Change this to the desired POS tag
pos_to_replace = [pos.strip() for pos in pos_to_replace.split(",")]


In [185]:


base_filename = Path(filename).stem

rules_filename = Path("extracted_rules") / f"{base_filename}_sections_classified_gpt_extracted_rules_direct_parsed.json"
all_rules,rules_with_fst,rules_with_unimorph = load_rules(rules_filename)

# Flattened list of all rules
print(f"No of rules: {len(all_rules)}")
print(f"No of rules with FST: {len(rules_with_fst)}")
print(f"No of rules with Unimorph: {len(rules_with_unimorph)}")
# Filter out rules into differnet categories (add tags?)
for rule in rules_with_unimorph:
    try:
        split_unimorph = rule["unimorph"].split(";")
        # print(split_unimorph)
    except:
        print(rule["unimorph"])

# Use the dynamically constructed parallel sentences filename

parallel_sentences_file = (
    Path("parallel_sents")
    / f"{base_filename}_parallel_sentences_with_pos_and_unimorph.json"
)

with open(parallel_sentences_file, "r", encoding="utf-8") as f:
    parallel_sentences_with_pos_and_unimorph = json.load(f)

# Load dictionary file based on base_filename if available, else use default path
dict_path = Path("dictionary") / f"{base_filename}_dictionary.json"
dictionary = load_dictionary(dict_path)
dictionary_with_metadata_path = Path("dictionary") / f"{base_filename}_dictionary_with_metadata.json"
dictionary_with_metadata = load_dictionary_with_meta(dictionary_with_metadata_path)

# TODO: Make this filtering for each POS
nouns_only_dict = filter_dict_by_pos(dictionary_with_metadata, target_pos=["NOUN", "PROPN"])
verbs_only_dict = filter_dict_by_pos(dictionary_with_metadata, target_pos=["VERB"])
adjectives_only_dict = filter_dict_by_pos(dictionary_with_metadata, target_pos=["ADJ"])
adverbs_only_dict = filter_dict_by_pos(dictionary_with_metadata, target_pos=["ADV"])
pos_dicts = {
    "NOUN": nouns_only_dict,
    "VERB": verbs_only_dict,
    "ADJ": adjectives_only_dict,
    "ADV": adverbs_only_dict,
}


No of rules: 812
No of rules with FST: 337
No of rules with Unimorph: 387


In [186]:
# Embed rules
import torch
embed_rules = False

if embed_rules == True:

    rule_texts = [
        f"{r.get('description','')} {r.get('condition','')} {' '.join(r.get('grammar_tags') or [])}"
        for r in all_rules
    ]
    rule_embs = model.encode(rule_texts, convert_to_tensor=True, normalize_embeddings=True)
    # Save to disk
    torch.save(rule_embs, "rule_embeddings.pt")
    with open("rule_texts.json", "w") as f: json.dump(rule_texts, f)

else:
    rule_embeddings = torch.load("rule_embeddings.pt")


In [187]:
no_of_random_nouns = 20
random_words_of_selected_pos = get_random_words_from_dict(
    pos_dicts, pos_to_replace, no_of_random_nouns
)
for word, pos_info in random_words_of_selected_pos.items():
    print(f"Random {pos_to_replace} word: {word} - {pos_info}")

Random ['NOUN', 'PNOUN'] word: mustika - {'senses': [{'translation': 'pearl shell', 'page_num': 530, 'cleaned_translation': 'pearl shell', 'grammatical_features': {}, 'spacy_analysis': {'is_multiword': True, 'pos_sequence': ['PROPN', 'NOUN'], 'tag_sequence': ['NNP', 'NN'], 'morph_sequence': [{'Number': 'Sing'}, {'Number': 'Sing'}], 'lemmas': ['pearl', 'shell'], 'text': ['pearl', 'shell'], 'head_index': 1, 'head_text': 'shell', 'head_pos': 'NOUN', 'head_lemma': 'shell', 'dependencies': [{'token': 'pearl', 'dep': 'amod', 'head': 'shell'}, {'token': 'shell', 'dep': 'ROOT', 'head': 'shell'}]}}]}
Random ['NOUN', 'PNOUN'] word: walaka - {'senses': [{'translation': 'Goromese', 'page_num': 524, 'cleaned_translation': 'Goromese', 'grammatical_features': {}, 'spacy_analysis': {'is_multiword': False, 'pos_sequence': ['NOUN'], 'tag_sequence': ['NN'], 'morph_sequence': [{'Number': 'Sing'}], 'lemmas': ['goromese'], 'text': ['Goromese'], 'head_index': 0, 'head_text': 'Goromese', 'head_pos': 'NOUN', '

In [188]:
base_prompt = f"""You are a professional linguist specializing in grammar-based sentence generation in the low-resource language {{lang}}. 
You are NOT allowed to simply substitute words — you must apply grammatical reasoning and cite the rules that justify every change.

---

### Incorrect behavior
Do NOT:
- blindly replace the target word with the new one
- keep the same morphology or clitic pattern without checking the rules
- output an ungrammatical or rule-violating sentence

---

### Correct behavior
You MUST:
- read and understand the rules carefully
- justify which rules apply, why, and how they affect morphology, clitics, or word order
- explain each grammatical transformation step-by-step
- only then generate the final new sentence

---

### INPUTS
- **Rules** (relevant to this POS and its transformations):
{{rules_data}}

- **Original sentence (transliterated {{lang}})**:
{{sentence_text}}

- **English translation**:
{{translated_sentence}}

- **POS + Unimorph metadata for each token**:
{{pos_unimorph_info}}

- **POS to be replaced**: {{pos}}

- **New word to use**: {{word}} — "{{word_translation}}"

---

### Task
Replace the {{pos}} in the sentence using the new word **only after** selecting and applying the correct grammatical rules.
Make sure the resulting sentence is grammatically correct, natural, and faithful to the language structure as described by the rules.

---

### Step-by-step reasoning plan (MANDATORY)
1. **Identify Target Context**
   - Which word in the original sentence will be replaced?
   - What is its current POS and morphological role (Case, Number, Gender, etc.)?

2. **Select Relevant Rules**
   - From the provided list, choose the rules that apply to this word or its syntactic environment.
   - For each chosen rule, cite the rule by name or description and briefly explain *why* it applies.
   - If a rule does *not* apply, briefly explain why not.

3. **Derive Transformation Plan**
   - Based on the chosen rules, describe:
     - what morphological or clitic changes must happen
     - any word-order or agreement adjustments required
     - whether the new word needs inflection (e.g. plural, accusative, postposition, etc.)

4. **Generate the New Sentence**
   - Apply the rules step-by-step.
   - Show the intermediate form (if helpful).
   - Finally, write the full new sentence in {{lang}} (transliterated to English).

5. **Provide the English Translation**
   - Translate the new {{lang}} sentence into English.

---

### Output Format
Follow this structure exactly:

Step 1: [your reasoning for identifying context]  
Step 2: [selected rules + justification for each]  
Step 3: [detailed transformation plan]  
Step 4: [new {{lang}} sentence, transliterated]  
Step 5: [English translation of new sentence]

Final Output:
"{{lang}} sentence"
"English translation"


---

### Additional Guidance
- Prefer grammatical accuracy and correct rule usage over literal meaning.
- If no rules clearly apply, state “No applicable rules — sentence left unchanged.”
- If multiple rules conflict, justify which one you prioritized and why.
- You may inflect or reorder words as needed — rule-constrained creativity is encouraged.
- Always cite which rules you used, and show the reasoning before the final output.

---

Now begin by identifying the target word and explaining which rules from above govern its replacement."""

In [189]:
from tqdm import tqdm
from pathlib import Path
import json

def generate_batch_json(
    random_words_of_selected_pos,
    rule_text,
    sentence_text,
    translated_sentence,
    pos_info_text,
    pos_to_replace,
    base_filename,
    output_dir,
    MODEL_PROVIDER="openai",
    send_to_api=False,
):
    batch_requests = []

    for word in tqdm(random_words_of_selected_pos.keys()):
        word = word.strip()
        word_translation = (
            random_words_of_selected_pos[word]
            .get("senses", [{}])[0]
            .get("translation", "No translation available")
        )

        if word_translation == "No translation available":
            print(f"⚠️  Warning: No translation found for word '{word}'")
            continue

        # Build the prompt
        prompt = base_prompt.format(
            rules_data=rule_text,
            sentence_text=sentence_text,
            translated_sentence=translated_sentence,
            pos_unimorph_info=pos_info_text,
            lang=base_filename.split("_")[0],  # Extract language code
            word=word,
            word_translation=word_translation,
            pos=", ".join(pos_to_replace),
        )

        # Store request entry
        batch_requests.append(
            {
                "word": word,
                "word_translation": word_translation,
                "prompt": prompt,
                "language": base_filename.split("_")[0],
                "pos": pos_to_replace,
            }
        )

    # Save all prompts to JSON file
    batch_file = Path(output_dir) / f"{base_filename}_batch_prompts.json"
    batch_file.parent.mkdir(parents=True, exist_ok=True)
    with open(batch_file, "w", encoding="utf-8") as f:
        json.dump(batch_requests, f, ensure_ascii=False, indent=4)
    print(f"✅ Saved {len(batch_requests)} prompts to {batch_file}")


    return batch_file



In [190]:
#########################################
# Main generation loop
#########################################


def generate_sentences(sentence_limit=400, filter_rules=True, VERBOSE=False, 
                       pos_to_replace=pos_to_replace, no_of_random_nouns=no_of_random_nouns):
    """
    Generate sentences by replacing words of specified POS using grammatical rules.

    Args:
        sentence_limit (int): Maximum number of sentences to process. Set to -1 for no limit.
        filter_rules (bool): Whether to filter rules based on morphology.
        VERBOSE (bool): Whether to print detailed logs.

    Returns:
        None
    """


    generated_sentences = []
    all_batch_requests = []



    LIMIT = min(max(sentence_limit, 0), len(parallel_sentences_with_pos_and_unimorph))
    print(f"Using {LIMIT} sentences for generation out of {len(parallel_sentences_with_pos_and_unimorph)}.")

    test_set = parallel_sentences_with_pos_and_unimorph[LIMIT:]  # Reserved for testing/evaluation
    # Save test set to a separate file
    test_set_file = Path("parallel_sents") / f"{base_filename}_test_set.json"
    with open(test_set_file, "w", encoding="utf-8") as f:
        json.dump(test_set, f, ensure_ascii=False, indent=4)    

    for idx, sentence in enumerate(parallel_sentences_with_pos_and_unimorph[:LIMIT]):

        # print(random_nouns)
        random_words_of_selected_pos = get_random_words_from_dict(
                pos_dicts, pos_to_replace, no_of_random_nouns
            )

        print("Processing sentence ", idx + 1)
        if VERBOSE:
            print(f"Selected sentence for generation {idx + 1}:")
            print(sentence["sentence"], "\n", sentence["translated_sentence"], "\n")
        if LIMIT <= 0:
            break
        LIMIT -= 1
        sentence_text = sentence["sentence"]
        translated_sentence = sentence["translated_sentence"]
        spacy_info_all = sentence["spacy_info"]
        spacy_info = spacy_info_all.get("res", None)
        tense = spacy_info_all.get("tense", None)
        number = spacy_info_all.get("number", None)
        genders = spacy_info_all.get("genders", None)

        if VERBOSE:
            print("Tense: ", tense)
            print(f"Number: {number}")
            print(f"Genders: {genders}")

        # Find LRL words by POS in the sentence
        lrl_words_by_pos = find_lrl_words_by_pos(
            sentence_text, pos_dicts, target_pos=pos_to_replace
        )
        #print(f"LRL words by POS in the sentence: {lrl_words_by_pos}")

        # Quick check for kalamang, remove "se" from nouns
        if base_filename == "kalamang":
            if "NOUN" in lrl_words_by_pos:
                lrl_words_by_pos["NOUN"] = [
                    w for w in lrl_words_by_pos["NOUN"] if w != "se"
                ]
                if lrl_words_by_pos["NOUN"] == []:
                    del lrl_words_by_pos["NOUN"]

        gender, plural, case = get_lrl_words_meta(lrl_words_by_pos, spacy_info, dictionary, sentence)

        ################
        # RULE SELECTION
        ################

        if VERBOSE:
            print("\nRule selection\n")

        if filter_rules:
            rule_text = select_rules(
                all_rules, gender=gender, number=number, case=case, tense=tense
            )
        else:
            rule_text = "\n".join("- " + rule["description"] for rule in all_rules)

        # Count the amount of " - " in rule_text
        if VERBOSE:
            print(rule_text)
            print("----------------------------------------------\n")
        ###############
        ###############
        ###############
        # ruleset_text = "\n".join("- " + rule["description"] for rule in ruleset)


        pos_info_text = ""
        for token in spacy_info:
            # Shorten morph info for readability
            morph_info = token.get("morph", {})
            morph_info = {k: v for k, v in morph_info.items() if k in ["Case", "Number", "Gender"]}
            # Shorten lemma for readability
            lemma = token['lemma']
            pos_info_text += f"- {lemma}: {morph_info}\n"

        if VERBOSE:
            print("----------------------------------------------\n")
            print(pos_info_text)
        ###############
        ###############
        ###############
        
        # --- Instead of calling generate_batch_json, build prompt entries directly ---
        for word in random_words_of_selected_pos.keys():
            word = word.strip()
            word_translation = (
                random_words_of_selected_pos[word]
                .get("senses", [{}])[0]
                .get("translation", "No translation available")
            )
            if word_translation == "No translation available":
                continue

            prompt = base_prompt.format(
                rules_data=rule_text,
                sentence_text=sentence_text,
                translated_sentence=translated_sentence,
                pos_unimorph_info=pos_info_text,
                lang=base_filename.split("_")[0],
                word=word,
                word_translation=word_translation,
                pos=", ".join(pos_to_replace),
            )
            all_batch_requests.append(
                {
                    "sentence_index": idx + 1,
                    "sentence_text": sentence_text,
                    "translated_sentence": translated_sentence,
                    "word": word,
                    "word_translation": word_translation,
                    "prompt": prompt,
                    "tense": tense,
                    "number": number,
                    "gender": genders,
                    "case": case,
                    "pos": pos_to_replace,
                }
            )

    # Save all batch requests to JSON file
    output_dir = Path("generation_prompts") / base_filename
    output_dir.mkdir(parents=True, exist_ok=True)
    batch_file = Path(output_dir) / f"{base_filename}_all_batch_prompts_{'-'.join(pos_to_replace)}_{sentence_limit}.json"
    with open(batch_file, "w", encoding="utf-8") as f:
        json.dump(all_batch_requests, f, ensure_ascii=False, indent=4)
    f.close()
    print(f"✅ Saved {len(all_batch_requests)} prompts to {batch_file}")


In [191]:
pos_to_replace_list = [["NOUN", "PNOUN"], ["VERB"], ["ADJ"], ["ADV"]]
sentence_limit = 400
no_of_random_nouns = 20
for pos_to_replace in pos_to_replace_list:
    generate_sentences(sentence_limit=sentence_limit, filter_rules=True, VERBOSE=False, 
                       pos_to_replace=pos_to_replace, no_of_random_nouns=no_of_random_nouns)

Using 400 sentences for generation out of 921.
Processing sentence  1
Words to be replaced: {'NOUN': ['bal']}
Translation: {'bal': 'dog'}
dog (bal) present in sentence
Selected word to be replaced: bal - dog
Found word stem info: {'word': 'dog', 'lemma': 'dog', 'upos': 'NOUN', 'tag': 'NN', 'morph': {'Number': 'Sing'}}
The word 'bal' is singular.
Processing sentence  2
No words found in the sentence to replace.
Processing sentence  3
Words to be replaced: {'NOUN': ['koup']}
Translation: {'koup': 'hug'}
hug (koup) present in sentence
Selected word to be replaced: koup - hug
Found word stem info: {'word': 'hugged', 'lemma': 'hug', 'upos': 'VERB', 'tag': 'VBD', 'morph': {'Tense': 'Past', 'VerbForm': 'Fin'}}
The word 'koup' is singular.
Processing sentence  4
No words found in the sentence to replace.
Processing sentence  5
Words to be replaced: {'NOUN': ['sor', 'me']}
Translation: {'sor': 'fish', 'me': 'topic marker ; distal demonstrative ; that'}
fish (sor) present in sentence
Selected wo

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import json
from tqdm import tqdm

def build_openai_batch_jsonl(
    all_batch_requests,
    output_dir="./outputs",
    base_filename="no_filename_provided",
    model="gpt-4o-mini",
):
    """
    Converts all_batch_requests into OpenAI batch-compatible .jsonl file
    """

    batch_lines = []
    for i, req in enumerate(tqdm(all_batch_requests, desc="Building OpenAI batch JSONL")):
        prompt = req["prompt"]

        # Create one valid batch request line
        entry = {
            "custom_id": f"{base_filename}_{i+1}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 1200,
                "temperature": 0.7,
            },
        }
        batch_lines.append(entry)

    # Save as JSONL (one object per line)
    batch_path = Path(output_dir) / f"{base_filename}_openai_batch.jsonl"
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    with open(batch_path, "w", encoding="utf-8") as f:
        for line in batch_lines:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")

    print(f"✅ Saved OpenAI batch file with {len(batch_lines)} requests to {batch_path}")
    print("You can now upload this file via the OpenAI batch endpoint.")
    return batch_path


In [ ]:
import os
all_batch_requests_file = Path("generation_prompts") / base_filename 

all_files = os.listdir(all_batch_requests_file)
for file in all_files:
    if file.startswith(f"{base_filename}_all_batch_prompts_") and file.endswith(".json"):
        with open(all_batch_requests_file / file, "r", encoding="utf-8") as f:
            all_batch_requests = json.load(f)
        pos_part = file.split("_all_batch_prompts_")[1].rsplit("_",1)[0]
        sentence_limit_part = file.rsplit("_",1)[1].replace(".json","")
        build_openai_batch_jsonl(
            all_batch_requests,
            output_dir=Path("generation_prompts") / base_filename,
            base_filename=file.replace(".json",""),
            model="gpt-4o-mini",
        )

Building OpenAI batch JSONL: 100%|██████████| 8000/8000 [00:00<00:00, 322921.33it/s]


✅ Saved OpenAI batch file with 8000 requests to generation_prompts\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch.jsonl
You can now upload this file via the OpenAI batch endpoint.


Building OpenAI batch JSONL: 100%|██████████| 8000/8000 [00:00<00:00, 232784.10it/s]


✅ Saved OpenAI batch file with 8000 requests to generation_prompts\kalamang\kalamang_all_batch_prompts_ADV_400_openai_batch.jsonl
You can now upload this file via the OpenAI batch endpoint.


Building OpenAI batch JSONL: 100%|██████████| 8000/8000 [00:00<00:00, 9616.45it/s]


✅ Saved OpenAI batch file with 8000 requests to generation_prompts\kalamang\kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch.jsonl
You can now upload this file via the OpenAI batch endpoint.


Building OpenAI batch JSONL: 100%|██████████| 8000/8000 [00:00<00:00, 144312.69it/s]


✅ Saved OpenAI batch file with 8000 requests to generation_prompts\kalamang\kalamang_all_batch_prompts_VERB_400_openai_batch.jsonl
You can now upload this file via the OpenAI batch endpoint.


In [ ]:
# Check jsonl and print number of lines
jsonl_files = os.listdir(all_batch_requests_file)
for file in jsonl_files:
    if file.endswith(".jsonl"):
        with open(all_batch_requests_file / file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        print(f"File {file} has {len(lines)} lines.")

File kalamang_all_batch_prompts_ADJ_400_openai_batch.jsonl has 8000 lines.
File kalamang_all_batch_prompts_ADV_400_openai_batch.jsonl has 8000 lines.
File kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch.jsonl has 8000 lines.
File kalamang_all_batch_prompts_VERB_400_openai_batch.jsonl has 8000 lines.


In [ ]:
# Split each file into 1600 line chunks if larger than 1600 lines
for file in jsonl_files:
    if file.endswith(".jsonl"):
        with open(all_batch_requests_file / file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        if len(lines) > 1600:
            chunk_size = 1600
            num_chunks = (len(lines) + chunk_size - 1) // chunk_size
            for i in range(num_chunks):
                chunk_lines = lines[i*chunk_size:(i+1)*chunk_size]
                chunk_file = all_batch_requests_file / f"{file.replace('.jsonl','')}_part{i+1}.jsonl"
                with open(chunk_file, "w", encoding="utf-8") as cf:
                    cf.writelines(chunk_lines)
                print(f"Created chunk file {chunk_file} with {len(chunk_lines)} lines.")
            # Optionally remove the original large file
           


Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch_part1.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch_part2.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch_part3.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch_part4.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch_part5.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADV_400_openai_batch_part1.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADV_400_openai_batch_part2.jsonl with 1600 lines.
Created chunk file generation_prompts\kalamang\kalamang_all_batch_prompts_ADV_400_openai_batch_part3.jsonl with 1600 lines.
Created 

In [ ]:
import os
import time
from pathlib import Path
from openai import OpenAI

# --- Load API key ---
with open("api_key.txt", "r") as f:
    client = OpenAI(api_key=f.read().strip())

# --- Folder with batch .jsonl files ---
batch_folder = Path("./generation_prompts")

# --- Output file to store batch IDs ---
batch_log_path = Path("./outputs/batch_job_ids.txt")
batch_log_path.parent.mkdir(parents=True, exist_ok=True)

# --- Batch settings ---
COMPLETION_WINDOW = "24h"
POLL_INTERVAL = 15 * 60  # 15 minutes
CONFIRM_SUBMIT = True

# --- Prepare log file ---
with open(batch_log_path, "w", encoding="utf-8") as log:
    log.write("# Batch Jobs Created\n")

# --- Helper function to wait for a batch to finish ---
def wait_for_completion(client, batch_id):
    while True:
        batch = client.batches.retrieve(batch_id)
        print(f"🔍 Batch {batch_id} status: {batch.status}")
        if batch.status in ("completed", "failed", "cancelled", "expired"):
            return batch.status
        print(f"⏳ Waiting {POLL_INTERVAL/60} minutes...")
        time.sleep(POLL_INTERVAL)

# --- Main loop ---
for file in sorted(os.listdir(batch_folder)):
    if file.endswith(".jsonl") and "_part" in file:
        batch_path = batch_folder / file
        with open(batch_path, "r", encoding="utf-8") as f:
            line_count = sum(1 for _ in f)

        print(f"📦 {file}: {line_count} requests")

        if not CONFIRM_SUBMIT:
            print("⚠️ Dry run mode — skipping upload.")
            continue

        # --- Upload and create batch ---
        uploaded = client.files.create(file=open(batch_path, "rb"), purpose="batch")
        batch_job = client.batches.create(
            input_file_id=uploaded.id,
            endpoint="/v1/chat/completions",
            completion_window=COMPLETION_WINDOW,
        )

        print(f"🚀 Submitted batch job {batch_job.id} for {file}")
        print(f"🕒 Initial status: {batch_job.status}\n")

        # --- Save to log ---
        with open(batch_log_path, "a", encoding="utf-8") as log:
            log.write(f"{file}\t{batch_job.id}\t{line_count} requests\n")

        # --- Wait until it finishes ---
        final_status = wait_for_completion(client, batch_job.id)
        print(f"✅ Batch {batch_job.id} finished with status: {final_status}\n")

print("🎉 All batch jobs processed sequentially!")
print(f"🧾 Log saved at: {batch_log_path}")


📦 kalamang_all_batch_prompts_ADJ_400_openai_batch_part1.jsonl: 1600 requests
🚀 Submitted batch job batch_68f6ac83ee208190be5cacf9d2964d77 for kalamang_all_batch_prompts_ADJ_400_openai_batch_part1.jsonl
🕒 Status: validating

📦 kalamang_all_batch_prompts_ADJ_400_openai_batch_part2.jsonl: 1600 requests
🚀 Submitted batch job batch_68f6ac87853c8190b0beeea7921a7215 for kalamang_all_batch_prompts_ADJ_400_openai_batch_part2.jsonl
🕒 Status: validating

📦 kalamang_all_batch_prompts_ADJ_400_openai_batch_part3.jsonl: 1600 requests
🚀 Submitted batch job batch_68f6ac8a4f908190954d66c5e49d229c for kalamang_all_batch_prompts_ADJ_400_openai_batch_part3.jsonl
🕒 Status: validating

📦 kalamang_all_batch_prompts_ADJ_400_openai_batch_part4.jsonl: 1600 requests
🚀 Submitted batch job batch_68f6ac8d18d08190956e3e33b5e7da74 for kalamang_all_batch_prompts_ADJ_400_openai_batch_part4.jsonl
🕒 Status: validating

📦 kalamang_all_batch_prompts_ADJ_400_openai_batch_part5.jsonl: 1600 requests
🚀 Submitted batch job batch

# Batch process test

In [ ]:
import os
import openai
import requests

def send_openai_batch_request(batch_file_path, api_key):
    """
    Sends a batch request to OpenAI using the provided batch file.
    Args:
        batch_file_path (str): Path to the batch .jsonl file.
        api_key (str): OpenAI API key.
    Returns:
        dict: Response from the OpenAI API.
    """
    url = "https://api.openai.com/v1/batches"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/jsonl",
    }

    with open(batch_file_path, "r", encoding="utf-8") as f:
        batch_data = f.read()

    response = requests.post(url, headers=headers, data=batch_data)

    if response.status_code == 200:
        print("✅ Batch request sent successfully.")
        return response.json()
    else:
        print(f"❌ Failed to send batch request. Status code: {response.status_code}")
        print(f"Response: {response.text}")
        return None

In [ ]:
# Batch send example
from openai import OpenAI

with open("api_key.txt", "r") as f:
    openai_api_key = f.read().strip()

client = OpenAI(api_key=openai_api_key)

results = []
for batch_request in all_batch_requests[20:40]:  # Send only first 20 for testing
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": batch_request["prompt"]}
        ]
    )
    print(f"Response for word '{batch_request['word']}':")
    print(response.choices[0].message.content)  
    results.append({
        "word": batch_request["word"],
        "response": response.choices[0].message.content
    })

print(f"✅ Completed sending {len(results)} requests.")

Response for word 'dagim':
Step 1: The words identified for replacement are "one hundred and fifty [thousand rupiah]" since they represent a noun phrase that serves as the object of the verb 'to send'. The original term is a compound expression in the accusative case, indicating the concept of currency. 

Step 2: The relevant rules that apply to the nouns being replaced include:

- The rules governing the cases: The pronoun "I" (the object case) and the noun phrase “one hundred and fifty [thousand rupiah]” need to be replaced correctly while maintaining their grammatical structure.
- The replacement noun "dagim" means "meat" and does not require any specific case modification since it will be taking the role of the object in similar grammatical contexts.
- The quantification structure of “one hundred and fifty thousand” is important to preserve, as it reflects a specific numeric item being sent.

Step 3: The transformation plan involves the following:

- Replace the entire noun phrase 

In [ ]:
# Parse and save results
import os


final_results_file = Path("generation_results") / f"{base_filename}_generation_results_{'-'.join(pos_to_replace)}_{sentence_limit}.json"
os.makedirs(final_results_file.parent, exist_ok=True)
with open(final_results_file, "w") as f:
    json.dump(results, f, indent=2)


NameError: name 'pos_to_replace' is not defined

In [ ]:
final_results = []
for res in results:
    # Find "Final Output:"
    content = res["response"]
    final_output_match = re.search(r'Final Output:\s*"(.*?)"\s*"(.*?)"', content, re.DOTALL)
    if final_output_match:
        lrl_sentence = final_output_match.group(1).strip()
        eng_translation = final_output_match.group(2).strip()
        print(f"Generated sentence for word '{res['word']}':")
        print(f"{lrl_sentence}")
        print(f"{eng_translation}\n")
        # Save the results
        final_results.append({
            "word": res["word"],
            "lrl_sentence": lrl_sentence,
            "eng_translation": eng_translation
        })





Generated sentence for word 'dagim':
ma reitkon dagim an=at kamat=et
He sent me meat.

Generated sentence for word 'nakafan':
ma reitkon purap-i an=at kamat=et nakafan=et
He sent me one hundred and fifty nakafan.

Generated sentence for word '40. 61–96.':
ma reitkon purap-i an=at 40. 61–96. — 'Linguistic studies of languages in and around Indonesia' =et
He sent me Linguistic studies of languages in and around Indonesia.

Generated sentence for word 'buok':
ma reitkon buok an=at kamat=et
He sent me betel.

Generated sentence for word 'bugar':
ma reitkon purap-i an=at kamat=et bugar.
He sent me one hundred and fifty fish.

Generated sentence for word 'akal':
ma reitkon purap-i an=at akal=et
He sent me sense.

Generated sentence for word 'nadou':
ma reitkon purap-i an=at nadou=et
He sent me a nod.

Generated sentence for word 'sanggien':
ma reitkon purap-i an=at kamat=et sanggien
He sent me one hundred and fifty birds of paradise.

Generated sentence for word 'tep':
ma reitkon purap-i kam

In [ ]:
# show final results
for res in final_results:
    print(f"Generated sentence for word '{res['word']}':")
    print(f"{res['lrl_sentence']}")
    print(f"{res['eng_translation']}\n")

Generated sentence for word 'dagim':
ma reitkon dagim an=at kamat=et
He sent me meat.

Generated sentence for word 'nakafan':
ma reitkon purap-i an=at kamat=et nakafan=et
He sent me one hundred and fifty nakafan.

Generated sentence for word '40. 61–96.':
ma reitkon purap-i an=at 40. 61–96. — 'Linguistic studies of languages in and around Indonesia' =et
He sent me Linguistic studies of languages in and around Indonesia.

Generated sentence for word 'buok':
ma reitkon buok an=at kamat=et
He sent me betel.

Generated sentence for word 'bugar':
ma reitkon purap-i an=at kamat=et bugar.
He sent me one hundred and fifty fish.

Generated sentence for word 'akal':
ma reitkon purap-i an=at akal=et
He sent me sense.

Generated sentence for word 'nadou':
ma reitkon purap-i an=at nadou=et
He sent me a nod.

Generated sentence for word 'sanggien':
ma reitkon purap-i an=at kamat=et sanggien
He sent me one hundred and fifty birds of paradise.

Generated sentence for word 'tep':
ma reitkon purap-i kam

In [ ]:

import os
from pathlib import Path


In [196]:
# Read openai batch output
from tqdm import tqdm
file_path = "outputs/completions/"

file_list = os.listdir(file_path)

result_data = {}

for file_name in file_list:
    if file_name.endswith(".jsonl"):
        print(f"Reading file: {file_name}")
        idx = file_name.find("part")
        if idx !=-1:
            file_name_base = file_name[:idx-1]
            print(f"File base name: {file_name_base}")
        
        if result_data.get(file_name_base) is None:
            result_data[file_name_base] = []
            #print(data)
    
        with open(os.path.join(file_path, file_name), "r", encoding="utf-8") as f:
            for line in tqdm(f):
                data = json.loads(line)
                response = data.get("response", {})
                body = response.get("body", {})
                choices = body.get("choices", [])
                custom_id = data.get("custom_id", "No ID")
                #print(response)
                if choices:
                    content = choices[0].get("message", {}).get("content", "")
                    
                    #print(content)
                    final_output_match = re.search(r'Final Output:\s*"(.*?)"\s*"(.*?)"', content, re.DOTALL)
                    if final_output_match:
                        lrl_sentence = final_output_match.group(1).strip()
                        eng_translation = final_output_match.group(2).strip()
                        #print(f"Generated sentence:")
                        #print(f"{lrl_sentence}")
                        #print(f"{eng_translation}\n")
                        result_data[file_name_base].append({
                            "custom_id": custom_id,
                            "generated_sentence": lrl_sentence,
                            "generated_eng_translation": eng_translation
                        })
                


Reading file: kalamang_all_batch_prompts_ADJ_400_openai_batch_part1.jsonl_output.jsonl
File base name: kalamang_all_batch_prompts_ADJ_400_openai_batch


1600it [00:00, 20106.32it/s]


Reading file: kalamang_all_batch_prompts_ADJ_400_openai_batch_part5.jsonl_output.jsonl
File base name: kalamang_all_batch_prompts_ADJ_400_openai_batch


1600it [00:00, 31829.44it/s]


Reading file: kalamang_all_batch_prompts_ADV_400_openai_batch_part1.jsonl_output.jsonl
File base name: kalamang_all_batch_prompts_ADV_400_openai_batch


...

1600it [00:00, 24284.10it/s]


Reading file: kalamang_all_batch_prompts_VERB_400_openai_batch_part4.jsonl_output.jsonl
File base name: kalamang_all_batch_prompts_VERB_400_openai_batch


1600it [00:00, 11652.54it/s]


Reading file: kalamang_all_batch_prompts_VERB_400_openai_batch_part5.jsonl_output.jsonl
File base name: kalamang_all_batch_prompts_VERB_400_openai_batch


1600it [00:00, 19900.80it/s]


In [197]:
result_data.keys()

dict_keys(['kalamang_all_batch_prompts_ADJ_400_openai_batch', 'kalamang_all_batch_prompts_ADV_400_openai_batch', 'kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch', 'kalamang_all_batch_prompts_VERB_400_openai_batch'])

In [198]:
# Store each result data item as a separate JSON file with name based on the key
output_results_path = Path("generation_results_combined") / base_filename
output_results_path.mkdir(parents=True, exist_ok=True)
for key, results in result_data.items():
    output_file = output_results_path / f"{key}_results.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
    print(f"Saved results to {output_file}")

Saved results to generation_results_combined\kalamang\kalamang_all_batch_prompts_ADJ_400_openai_batch_results.json
Saved results to generation_results_combined\kalamang\kalamang_all_batch_prompts_ADV_400_openai_batch_results.json
Saved results to generation_results_combined\kalamang\kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results.json
Saved results to generation_results_combined\kalamang\kalamang_all_batch_prompts_VERB_400_openai_batch_results.json


In [214]:
import json
from pathlib import Path

# --- Configuration ---
INPUT_DIR = Path("generation_results_combined/kalamang/")             # Folder with .json files
OUTPUT_DIR = Path("generation_results_combined_lemma/kalamang")   # Output folder
GROUP_SIZE = 20                             # Each set size (e.g., 20)
TAKE_COUNTS = [20]                       # How many to take from each segment
OFFSET_STEP = 20                            # Jump size for next block (usually same as GROUP_SIZE)

# Create output folder if missing
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"🔍 Processing JSON files in {INPUT_DIR}...")

files = os.listdir(INPUT_DIR)
print(files)
print(f"Found {len(files)} files.")

for json_file in files:
    json_file = Path(json_file)
    print(f"📄 Processing {json_file}")

    # --- Read JSON file ---
    with open(INPUT_DIR / json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        print(f"⚠️ Skipping {json_file.name}: not a list")
        continue

    selected = []

    # --- Loop through the list in groups ---
    for i in range(0, len(data), GROUP_SIZE):
        # Example: if TAKE_COUNTS = [5, 10], we’ll take first 5, then next 10
        start = i
        for take in TAKE_COUNTS:
            selected.extend(data[start:start + take])
            start += OFFSET_STEP  # move 20 forward for the next block

    # --- Save filtered data ---
    output_path = OUTPUT_DIR / f"{json_file.stem}_{TAKE_COUNTS[0]}_subset.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(selected, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(selected)} items to {output_path.name}\n")


🔍 Processing JSON files in generation_results_combined\kalamang...
['kalamang_all_batch_prompts_ADJ_400_openai_batch_results.json', 'kalamang_all_batch_prompts_ADV_400_openai_batch_results.json', 'kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results.json', 'kalamang_all_batch_prompts_VERB_400_openai_batch_results.json']
Found 4 files.
📄 Processing kalamang_all_batch_prompts_ADJ_400_openai_batch_results.json
✅ Saved 3154 items to kalamang_all_batch_prompts_ADJ_400_openai_batch_results_20_subset.json

📄 Processing kalamang_all_batch_prompts_ADV_400_openai_batch_results.json
✅ Saved 7861 items to kalamang_all_batch_prompts_ADV_400_openai_batch_results_20_subset.json

📄 Processing kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results.json
✅ Saved 7838 items to kalamang_all_batch_prompts_NOUN-PNOUN_400_openai_batch_results_20_subset.json

📄 Processing kalamang_all_batch_prompts_VERB_400_openai_batch_results.json
✅ Saved 7877 items to kalamang_all_batch_prompts_VERB_400_op

In [ ]:
all_results = []

for res in batch_results:
    response = res.get("response", {})
    body = response.get("body", {})
    choices = body.get("choices", [])
    custom_id = res.get("custom_id", "No ID")
    #print(response)
    if choices:
        content = choices[0].get("message", {}).get("content", "")
        print(f"Response for {custom_id}:")
        print(content)
        final_output_match = re.search(r'Final Output:\s*"(.*?)"\s*"(.*?)"', content, re.DOTALL)
        if final_output_match:
            lrl_sentence = final_output_match.group(1).strip()
            eng_translation = final_output_match.group(2).strip()
            print(f"Generated sentence:")
            print(f"{lrl_sentence}")
            print(f"{eng_translation}\n")
            all_results.append({
                "custom_id": custom_id,
                "lrl_sentence": lrl_sentence,
                "eng_translation": eng_translation
            })  
            


Response for kalamang_sample_all_NOUN-PNOUN_80_1:
Step 1: The target words for replacement in the original sentence are "dog" and "fish." Both of these words are singular nouns. In the context of the sentence, "dog" serves as the subject, and "fish" serves as the object of the verb "bite." The current POS for both is NOUN, and they are both in the nominative and accusative cases, respectively.

Step 2: The relevant rules selected for this transformation are:
- The expressions in the narrative are developed with fixed structures for songs. (This implies that the sentence structure should maintain its narrative form.)
- A dual pronoun followed by a noun with =bon refers to two people only. (While this rule does not apply directly, it suggests that if we were to refer to two entities, we would need to consider the structure of the sentence. In our case, we are not creating a dual form.)
- No other rules directly pertain to the nouns "dog" and "fish" in this context.

Step 3: The transform

In [ ]:
all_results

[{'custom_id': 'kalamang_sample_all_NOUN-PNOUN_80_1',
  'lrl_sentence': 'bal se sor=at koraru dagim iam meat=obj bite',
  'eng_translation': 'The meat has bitten the fish.'},
 {'custom_id': 'kalamang_sample_all_NOUN-PNOUN_80_2',
  'lrl_sentence': 'bal se sor=at koraru nakafan iam nakafan=obj bite',
  'eng_translation': 'The cloth has bitten the cloth.'},
 {'custom_id': 'kalamang_sample_all_NOUN-PNOUN_80_3',
  'lrl_sentence': 'bal se sor=at koraru 40. 61–96. iam fish=obj bite',
  'eng_translation': '[stim2_3:45] ‘The linguistic studies of languages in and around Indonesia have bitten the fish.’'},
 {'custom_id': 'kalamang_sample_all_NOUN-PNOUN_80_4',
  'lrl_sentence': 'bal se sor=at koraru buok iam buok=obj bite',
  'eng_translation': 'The betel has bitten the betel.'},
 {'custom_id': 'kalamang_sample_all_NOUN-PNOUN_80_5',
  'lrl_sentence': 'bal se sor=at koraru dog iam bugar=obj bite',
  'eng_translation': 'The dog has bitten the fish.'},
 {'custom_id': 'kalamang_sample_all_NOUN-PNOUN_

In [ ]:
# Save block table to a JSON file
output_file = "generation_results/final_generation_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)


# NLLB dataprep

In [ ]:
english_sentences = []
translated_sentences = []

for sentence in parallel_sentences_with_pos_and_unimorph:
    eng = sentence["translated_sentence"]
    source = sentence["sentence"]

    if eng!="" and source!="":
        english_sentences.append(eng)
        translated_sentences.append(source)


    

In [ ]:
english_sentences[:5], translated_sentences[:5]

(['[stim2_3:45] ‘The dog has bitten the fish.’',
  '‘He sent me one hundred and fifty [thousand rupiah].’ [conv12_3:09]',
  '‘They hugged the scared woman.’ [elic_adj19_8]',
  '‘I already wanted to sleep.’ [narr32_0:18]',
  '‘He said: “Hey, these fish, how [should we treat them]? Do we sell them, or do we split and salt them?”’ [narr8_5:34]'],
 ['bal se sor=at koraru dog iam fish=obj bite',
  'ma reitkon purap-i an=at kamat=et',
  'mu pas sem=ten=at koup',
  'an se toni min=kin',
  'toni: “Eh, sor wa me tamandi, pi parinet ye, pi parairet, Ma ma toni eh sor wa me tamandi pi parin=et ye pi parair=et siraet.” sira=et salt=irr'])

In [ ]:
# Store english sentences as eng.txt and translated sentences as lrl.txt

if os.path.exists("translation_data") == False:
    os.makedirs("translation_data")
with open("translation_data/eng.txt", "w", encoding="utf-8") as f:
    for line in english_sentences:
        f.write(line + "\n")

with open("translation_data/ckb.txt", "w", encoding="utf-8") as f:
    for line in translated_sentences:
        f.write(line + "\n")

# Test

In [ ]:
import json

filename = "generation_results\kalamang_generation_results_NOUN-PNOUN_80.json"
with open(filename, encoding="utf-8") as f:
    data = json.load(f)



<>:3: SyntaxWarning: invalid escape sequence '\k'
<>:3: SyntaxWarning: invalid escape sequence '\k'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_12040\2608495006.py:3: SyntaxWarning: invalid escape sequence '\k'
  filename = "generation_results\kalamang_generation_results_NOUN-PNOUN_80.json"


[{'word': 'dagim',
  'response': 'Step 1: The words identified for replacement are "one hundred and fifty [thousand rupiah]" since they represent a noun phrase that serves as the object of the verb \'to send\'. The original term is a compound expression in the accusative case, indicating the concept of currency. \n\nStep 2: The relevant rules that apply to the nouns being replaced include:\n\n- The rules governing the cases: The pronoun "I" (the object case) and the noun phrase “one hundred and fifty [thousand rupiah]” need to be replaced correctly while maintaining their grammatical structure.\n- The replacement noun "dagim" means "meat" and does not require any specific case modification since it will be taking the role of the object in similar grammatical contexts.\n- The quantification structure of “one hundred and fifty thousand” is important to preserve, as it reflects a specific numeric item being sent.\n\nStep 3: The transformation plan involves the following:\n\n- Replace the 

In [12]:
print(len(data))

20


In [11]:
count = 10

ignore = 20

for item in data:
    if ignore>0:
        ignore-=1
        continue
    word = item.get("word",None)
    print(f"Word: {word}")
    response = item.get("response",None)
    print(response)
    count-=1
    if(count<=0):
        break